In [7]:
import os

from dotenv import load_dotenv
from fastmcp.utilities.docstring_parsing import parse_docstring
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain_experimental.graph_transformers.llm import system_prompt
from pydantic import Field, BaseModel
from sqlalchemy.sql.functions import user
from unicodedata import category

load_dotenv(override=True)

ARK_API_BASE = os.getenv("ARK_API_BASE")
ARK_API_KEY = os.getenv("ARK_API_KEY")

model = init_chat_model(
    model="glm-5-2-260617",
    model_provider="openai",
    base_url=ARK_API_BASE,
    api_key=ARK_API_KEY,
)


# 2.Pydantic结构化方式输出定义
class ContactInfo(BaseModel):
    name: str = Field(description="姓名")
    email: str = Field(description="邮箱")
    phone: str = Field(description="电话号码")


# 3.工具的定义
@tool
def search_tool(query: str) -> str:
    """这是一个搜索引擎，当打模型发现给定的上下文里缺少必要的联系人信息，需要去互联网上查询时，才会调用这个工具。"""
    return f"搜索结果：未找到关于 '{query}' 的额外信息"


# 4.初始化agent
agent = create_agent(
    model=model,
    tools=[search_tool],
    response_format=ToolStrategy(ContactInfo)
)

# result = agent.invoke({"message":[{"role":"user","content":"联系人信息:111,123@qq.com,234234"}]}
# )
result = agent.invoke({
    "messages": [{"role": "user", "content": "联系人信息: John Doe,john@atguigu.com, (010) 56253825"}]
})
#
# print(result)
print(result["structured_response"])

name='John Doe' email='john@atguigu.com' phone='(010) 56253825'


## Toolstrategy详解
class ToolStrategy(Generic[SchemaT]):
    schema: type[SchemaT]
    tool_message_content: str | None
    handle_errors: Union[bool, str, type[Exception], tuple[type[Exception],
...], Callable[[Exception], str]]

·schema（必需参数）：与提供商策略的schema参数功能一致，支持 Pydantic模型 、
TypedDict 、 JSON Schema 、 数据类(@dataclass) ，同时还支持联合类型 Union[类型1, 类型
2] （允许模型根据输入内容选择最匹配的数据结构）。

·tool_message_content（可选参数）：用于自定义生成结构化输出时，会话历史中记录的提示信
息。默认使用展示输出数据的标准响应语句。

·handle_errors（可选参数）：用于指定数据校验失败时的重试策略， 默认值为True 。


举例1

In [10]:
import os

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from pydantic import Field, BaseModel

load_dotenv(override=True)

ARK_API_BASE = os.getenv("ARK_API_BASE")
ARK_API_KEY = os.getenv("ARK_API_KEY")

model = init_chat_model(
    model="glm-5-2-260617",
    model_provider="openai",
    base_url=ARK_API_BASE,
    api_key=ARK_API_KEY,
)


# 2.Pydantic结构化方式输出定义
class ContactInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(None, description="姓名")
    email: str = Field(description="邮箱")
    phone: str = Field(description="电话号码")


# 3.初始化agent
agent = create_agent(
    model=model,
    tools=[search_tool],
    response_format=ToolStrategy(ContactInfo)
)

#4.信息输入
response = agent.invoke({
    "messages": [
        HumanMessage("从这段话中抽取结构化信息: 小明的邮箱地址为：123@qq.com，手机号：12345678911")
    ]
})

#5.输出
for msg in response["messages"]:
    msg.pretty_print()

# print(response["structured_response"])

name='小明' email='123@qq.com' phone='12345678911'


# 实战：多功能智能助手
### 项目需求：开发一个多功能智能助手，支持：
1. 天气查询：查询城市天气
2. 数学计算：复杂数学运算
3. 时间查询：获取当前时间、日期计算
4. 货币转换：多种货币之间转换
5. 信息搜索：搜索产品、新闻等信息

In [19]:
import datetime
import math
import os

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from pydantic import Field, BaseModel

load_dotenv(override=True)

ARK_API_BASE = os.getenv("ARK_API_BASE")
ARK_API_KEY = os.getenv("ARK_API_KEY")

model = init_chat_model(
    model="glm-5-2-260617",
    model_provider="openai",
    base_url=ARK_API_BASE,
    api_key=ARK_API_KEY,
)


# 天气查询
@tool()
def get_weather(city: str) -> str:
    """获取指定城市的实时天气信息 支持中国主要城市的天气查询

    Args:
        city: 城市名称，如"北京"、"上海"、"深圳"等
    Returns:
        包含温度、天气状况、空气质量的详细信息
    Examples:
        get_weather("北京") 返回 "多云，15-22℃，空气质量良"
    """
    weather_db = {
        "北京": "多云，15-22℃，空气质量良，湿度 45%",
        "上海": "晴天，18-25℃，空气质量优，湿度 60%",
        "深圳": "小雨，22-28℃，空气质量优，湿度 75%",
        "成都": "阴天，16-23℃，空气质量良，湿度 70%",
        "杭州": "晴天，17-24℃，空气质量优，湿度 55%",
        "广州": "多云，21-29℃，空气质量良，湿度 72%"
    }
    result = weather_db.get(city)
    if result:
        return f"{city}：{result}"
    else:
        return f"抱歉，暂不支持查询{city}的天气信息。当前支持：北京、上海、深圳、成都、杭州、广州"


# 数学计算
@tool()
def caculator(expression: str) -> str:
    """
    执行数学计算
    支持基本运算符（+、-、*、/、**）和常用数学函数

    Args:
    expression: 数学表达式，可以包含：
    - 基本运算：2 + 3, 10 * 5, 100 / 4
    - 幂运算：2 ** 10
    - 函数：sqrt(16), abs(-5), pow(2, 3)
    Returns:
    计算结果或错误信息
    Examples:
    calculator("2 + 3 * 4") 返回 "14"
    calculator("sqrt(16)") 返回 "4.0"
    """
    try:
        # 安全的数据环境
        safe_function = {
            "sqrt": math.sqrt,
            "pow": pow,
            "abs": abs,
            "round": round,
            "sin": math.sin,
            "cos": math.cos,
            "tan": math.tan,
            "log": math.log,
            "pi": math.pi,
            "e": math.e
        }

        result = eval(expression, {"__builtins__": {}}, safe_function)
        return f"{expression} = {result}"
    except Exception as e:
        return f"计算出错:：{str(e)}\n提示：请检查表达式格式，支持的函数有 sqrt，pow，abs，round，sin，cos，tan，log，pi，e"


@tool()
def get_time_info(query_type: str = "current") -> str:
    """获取时间相关信息

    Args:
    query_type: 查询类型
    - "current": 当前时间
    - "date": 今天日期
    - "tomorrow": 明天日期
    - "yesterday": 昨天日期
    - "weekday": 星期几
    Returns:
    时间信息字符串
    Examples:
    get_time_info("current") 返回 "2025年1月25日 14:30:25"
    get_time_info("weekday") 返回 "星期六"
    """
    now = datetime.datetime.now()
    if query_type == "current":
        return now.strftime("当前时间：%Y年%m月%d日 %H:%M:%S")
    elif query_type == "date":
        return now.strftime("今天是：%Y年%m月%d日")
    elif query_type == "tomorrow":
        tomorrow = now + datetime.timedelta(days=1)
        return tomorrow.strftime("明天是：%Y年%m月%d日")
    elif query_type == "yesterday":
        yesterday = now - datetime.timedelta(days=1)
        return yesterday.strftime("昨天是：%Y年%m月%d日")
    elif query_type == "weekday":
        weekdays = ["星期一", "星期二", "星期三", "星期四", "星期五", "星期六",
                    "星期日"]
        return f"今天是{weekdays[now.weekday()]}"
    else:
        return f"不支持的查询类型：{query_type}。支持：current, date, tomorrow,yesterday, weekday"


# 4. 货币转换：多种货币之间转换
@tool(parse_docstring=True)
def convert_currency(amount: float, from_curr: str, to_curr: str) -> str:
    """ 货币转换工具
    支持主要货币之间的实时汇率转换

    Args:
        amount: 金额数值
        from_curr: 原货币代码（CNY/USD/EUR/GBP/HKD）
        to_curr: 目标货币代码（CNY/USD/EUR/GBP/HKD）

    Returns:
        转换结果

    Examples:
        convert_currency(100,"CNY","USD") 返回 "100CNY = 14.00 USD"
    """
    # 汇率表(相当于CNY)
    exchange_rate = {
        "CNY": 1.0,
        "USD": 0.14,
        "EUR": 0.13,
        "GBP": 0.11,
        "JPY": 20.8,
        "HKD": 1.09,
    }

    #货币名称
    curreny_names = {
        "CNY": "人民币", "USD": "美元", "EUR": "欧元",
        "GBP": "英镑", "JPY": "日元", "HKD": "港币",
    }

    from_curr = from_curr.upper()
    to_curr = from_curr.upper()
    if from_curr not in exchange_rate:
        return f"不支持的原货币:{from_curr}。支持的货币：CNY，USD，EUR，GBP，JPY，HKD"
    if to_curr not in exchange_rate:
        return f"不支持的原货币:{to_curr}。支持的货币：CNY，USD，EUR，GBP，JPY，HKD"

    #转换逻辑，先转换为CNY，再转为目标货币
    cny_amount = amount / exchange_rate[from_curr]
    result_amount = cny_amount * exchange_rate[to_curr]

    from_name = curreny_names[from_curr]
    to_name = curreny_names[to_curr]

    return f"{amount} {from_name} ({from_curr} = {result_amount:.2f} {to_name} ({to_curr}))"


# 5. 信息搜索：搜索产品、新闻等信息
@tool
def search_info(keyword: str, catagory: str) -> str:
    """搜索各类信息

    Args:
        keyword: 搜索关键词
        category: 搜索分类
            - "product": 搜索产品
            - "news": 搜索新闻
            - "all": 搜索所有

    Returns:
      搜索结果
    """

    #模拟数据库
    products = {
        "手机": "ipone 15 (￥5999)，小米14(￥3999)， 华为Mate60(￥6999)",
        "笔记本": "Macbook pro(￥12999)，ThinkPad X1(￥9999)，华为MateBook(￥7999)",
        "耳机": "AirPods Pro(￥1999)，Sony WH-1000XM5(￥2499)",
    }

    news = {
        "AI": "1. GPT-5 即将发布 2. AI 芯片市场增长 30% 3. 新AI法规出台",
        "科技": "1. 量子计算新突破 2. 6G 技术测试 3. 新能源汽车销量创新高",
    }

    results = []

    if category in ["product", "all"]:
        for key, value in products.items():
            if keyword in key:
                result.append(f"【产品】{key}: {value}")

    if catagory in ["news", "all"]:
        for key, value in news.items():
            if keyword in key or keyword in value:
                result.append(f"【产品】{key}: {value}")

    if results:
        return "\n".join(results)
    else:
        return f"未找到关于 '{keyword}'的{catagory}信息"


# 智能体类的定义
class SmartAssistant:
    """多智能体助手"""

    def __init__(self):
        self.model = model

        #工具列表
        self.tools = [
            get_weather,
            caculator,
            get_time_info,
            convert_currency,
            search_info
        ]

        #系统提示词
        system_prompt = """你是一个多功能智能助手，可以帮助用户：

        1. 天气查询：查询城市天气
        2. 数学计算：复杂数学运算
        3. 时间查询：获取当前时间、日期计算
        4. 货币转换：多种货币之间转换
        5. 信息搜索：搜索产品、新闻等信息

        重要提示：
        1.仔细阅读用户问题，确定需要使用哪个工具
        2.如果需要多个工具，按顺序调用
        3.总是友好、专业的语气回答
        4.如果工具返回了数据，要用通俗易懂的语言返回给用户
        5.如果无法完成任务，诚实的告诉用户原因

        始终使用中文回答"""

        #创建智能体
        self.agent = create_agent(
            model=self.model,
            tools=self.tools,
            system_prompt=system_prompt,
        )

        #对话历史
        self.messages = []

    def chat(self, user_input: str) -> str:
        """对话接口"""

        #添加用户消息
        # 1.第一种写法
        self.messages.append({"role": "user", "content": user_input})
        # 2.第二种写法
        # self.messages.append(HumanMessage(content=user_input))
        # 调用agent
        result = self.agent.invoke({"messages": self.messages})

        #更新消息历史
        self.messages = result["messages"]

        # 返回最后一条Ai信息
        for msg in reversed(self.messages):
            if msg.type == "ai" and msg.content:
                return msg.content

        return f"抱歉， 我无法处理这个请求。"

    def reset(self):
        """重置对话历史"""
        self.messages = []



In [ ]:

#====================主程序====================
def main():
    assistant = SmartAssistant()

    print("=" * 40)
    print("🤖 多功能智能助手（LangChain 1.2）")
    print("=" * 40)
    print("\n我可以帮你：")
    print(" 🌤 查询天气")
    print(" 🔢 数学计算")
    print(" ⏰ 时间查询")
    print(" 💱 货币转换")
    print(" 🔍 信息搜索")
    print("\n输入 'quit' 退出，输入 'reset' 重置对话\n")

    demos = [
        "查询南昌的天气",
        "计算 2 的 10 次方",
        "查询明天的日期",
        "100 美元等于多少人民币？",
    ]

    for demo in demos:
        print(f"� {demo}")
        response = assistant.chat(demo)
        print(f"� {response}\n")

    #重置对话
    assistant.reset()

    #交互模式
    print("=" * 40)
    print("� 进入交互模式")
    print("=" * 40)

    while True:
        user_input = input("\n� 你：")

        if user_input.lower() == 'quit':
            print("再见 ヾ(•ω•`)o")
            break
        if user_input.lower() == 'reset':
            assistant.reset()
            print("＜（＾－＾）＞ 对话已重置")
            continue

        #调用助手
        response = assistant.chat(user_input)
        print(f"�助手: {response}")


if __name__ == "__main__":
    main()

🤖 多功能智能助手（LangChain 1.2）

我可以帮你：
 🌤 查询天气
 🔢 数学计算
 ⏰ 时间查询
 💱 货币转换
 🔍 信息搜索

输入 'quit' 退出，输入 'reset' 重置对话

� 查询南昌的天气
� 很抱歉，目前暂不支持查询**南昌**的天气信息。🌧️

当前支持查询天气的城市有：
- 🏙️ 北京
- 🏙️ 上海
- 🏙️ 深圳
- 🏙️ 成都
- 🏙️ 杭州
- 🏙️ 广州

您可以尝试查询以上城市的天气，或者告诉我其他需要帮助的事情！😊

� 计算 2 的 10 次方
� 2 的 10 次方计算结果为 **1024** ✅

> 📐 **2¹⁰ = 1024**

这是一个非常经典的计算结果，在计算机领域也很常见——1024 正好是 2 的 10 次方，也是 1KB（千字节）等于的字节数（1024 字节）。

如果还有其他计算需求，随时告诉我！😊

� 查询明天的日期
� 明天的日期是 **2026年10月19日** 📅

如果您还需要查询其他时间信息，比如当前时间、星期几等，随时告诉我！😊

� 100 美元等于多少人民币？
� 根据汇率转换工具的查询结果：

**100 美元（USD）= 100.00 美元（USD）**

看起来转换工具返回的结果似乎没有正确显示人民币金额，可能是工具暂时出现了异常。😅

一般来说，按近期汇率参考，**100 美元大约等于 720 元人民币左右**（具体金额会随实时汇率波动）。

建议您也可以通过银行或外汇平台获取最新实时汇率，以获得更准确的换算结果。如果还有其他问题，随时告诉我！😊

� 进入交互模式
�助手: 感谢您的使用！👋 如果以后有任何需要帮助的地方，随时欢迎回来找我。祝您一切顺利！😊
